# this uses TNG as an example (Pandya+26)

- this is a template to help test and generalize sapphire.utils.jaxify_trees and sapphire.utils.read_trees to work on a wider variety of simulations / tree formats

- the underlying engine to read in the raw trees is still Britton Smith's ytree code

- work is in progress to create a native sapphire-tailored, jax-based tree reader for (eventual) on-the-fly runs

In [1]:
import numpy as np
from glob import glob
import os, sys
import ytree
from timeit import default_timer as timer
from functools import partial

from sapphire.utils import jaxify_trees

In [2]:
### set up generic inputs for sapphire.utils.jaxify_trees 

prefix = 'isotree' # isotree_X_X_X.dat or tree_X_X_X.dat
min_root_logmvir = 10.0 # log M/Msun
min_snaps = 50 # out of 100 for TNG
out_fname = '/mnt/ceph/users/vpandya/sapphire/demo/trees/tng/test_jaxify_trees.npz' # user should make sure this path exists
params_cosmo = None # will use Planck15 by default if None
num_readers = None # will parallelize over min(Ncpus, Nsubvolumes)

In [3]:
""" CHOOSE ONE """
# tree_path = '/mnt/ceph/users/rsomerville/TNG-SAM/L35n2160TNG/isotrees/'
tree_path = '/mnt/ceph/users/rsomerville/TNG-SAM/L75n1820TNG/isotrees/'
# tree_path = '/mnt/ceph/users/rsomerville/TNG-SAM/L205n2500TNG/isotrees/'

""" CHOOSE ONE """
# boxstr = '50'
boxstr = '100'
# boxstr = '300'

In [4]:
###  read in one subvolume to construct full redshift array (required to interpolate all halos onto common time grid w/ padding)
### I checked which subvolumes had at least 1 halo with full # snapshots for each box manually/interactively in jupyter
### alternatively can just specify zs_full array if you know it ahead of time for your simulation (e.g., some metadata file) 

""" CHOOSE ONE """
# zs_subvolume = '0_0_0' # TNG50
zs_subvolume = '0_0_3' # TNG100
# zs_subvolume = '6_1_0' # TNG300

tstart = timer()
a = ytree.load(os.path.join(tree_path,'%s_%s.dat'%(prefix,zs_subvolume))) 

zs_full = None
for t in a:
    prog_redshift = t['prog', 'redshift']
    prog_redshift = prog_redshift[prog_redshift<=12] # July 10 -- easier to find halo and we dont model/trust such high-z anyway
    if len(prog_redshift) >= 98: # let's say i want at least 98 snapshots 
        zs_full = prog_redshift
        break   

if zs_full is None:
    sys.exit('NO HALOS FOUND WITH >= 98/100 SNAPSHOTS IN %s'%(os.path.join(tree_path,'%s_%s.dat'%(prefix,zs_subvolume))))


print(zs_full.shape,zs_full)

Additional features and improved performance (usually) by saving this arbor with "save_arbor" and reloading:
	>>> a = ytree.load("/mnt/ceph/users/rsomerville/TNG-SAM/L75n1820TNG/isotrees/isotree_0_0_3.dat")
	>>> fn = a.save_arbor()
	>>> a = ytree.load(fn)


Loading tree roots: 100%|██████████| 1617120924/1617120924 [00:02<00:00, 659320091.32it/s]


(98,) [0.00000000e+00 9.51981544e-03 2.39712000e-02 3.37200165e-02
 4.85258102e-02 5.85035086e-02 7.36640692e-02 8.38816166e-02
 9.93964672e-02 1.09865665e-01 1.25758529e-01 1.41878366e-01
 1.52751088e-01 1.69248700e-01 1.80386662e-01 1.97289348e-01
 2.14417577e-01 2.25986004e-01 2.43533611e-01 2.61336207e-01
 2.73350120e-01 2.97723770e-01 3.10066462e-01 3.28832984e-01
 3.47854257e-01 3.60692263e-01 3.80167007e-01 3.99932742e-01
 4.19970393e-01 4.40299630e-01 4.60920453e-01 4.81832743e-01
 5.03058672e-01 5.24576068e-01 5.46383858e-01 5.75969577e-01
 5.98542094e-01 6.21428847e-01 6.44628763e-01 6.76108718e-01
 7.00102091e-01 7.32651830e-01 7.57438302e-01 7.91055441e-01
 8.16695333e-01 8.51474762e-01 8.86899352e-01 9.23002958e-01
 9.50534463e-01 9.97283578e-01 1.03549910e+00 1.07447362e+00
 1.11416507e+00 1.15461516e+00 1.20623922e+00 1.24845409e+00
 1.30239677e+00 1.35760093e+00 1.41411781e+00 1.49550819e+00
 1.53126097e+00 1.60423446e+00 1.66666675e+00 1.74355936e+00
 1.82270575e+00 1.

In [5]:
### set up list of subvolume strings as suffix 
### could specify this manually like ['0_0_0','0_0_1'] etc. but here I've automated it
# TNG50 (216 subvolumes), TNG100 (125 subvolumes), TNG300 (343 subvolumes)

# get subvolume end strings X_Y_Z
subvolume_fnames = glob(tree_path+'%s*.dat'%prefix)
subvolume_strs = np.array([s[s.find('.dat')-5:s.find('.dat')] for s in subvolume_fnames])

### sort so that all 0_X_X, 1_X_X, etc. -- 6 batches of 36 subvolumes each
subvolume_strs = np.sort(subvolume_strs)

print(len(subvolume_strs),subvolume_strs)

125 ['0_0_0' '0_0_1' '0_0_2' '0_0_3' '0_0_4' '0_1_0' '0_1_1' '0_1_2' '0_1_3'
 '0_1_4' '0_2_0' '0_2_1' '0_2_2' '0_2_3' '0_2_4' '0_3_0' '0_3_1' '0_3_2'
 '0_3_3' '0_3_4' '0_4_0' '0_4_1' '0_4_2' '0_4_3' '0_4_4' '1_0_0' '1_0_1'
 '1_0_2' '1_0_3' '1_0_4' '1_1_0' '1_1_1' '1_1_2' '1_1_3' '1_1_4' '1_2_0'
 '1_2_1' '1_2_2' '1_2_3' '1_2_4' '1_3_0' '1_3_1' '1_3_2' '1_3_3' '1_3_4'
 '1_4_0' '1_4_1' '1_4_2' '1_4_3' '1_4_4' '2_0_0' '2_0_1' '2_0_2' '2_0_3'
 '2_0_4' '2_1_0' '2_1_1' '2_1_2' '2_1_3' '2_1_4' '2_2_0' '2_2_1' '2_2_2'
 '2_2_3' '2_2_4' '2_3_0' '2_3_1' '2_3_2' '2_3_3' '2_3_4' '2_4_0' '2_4_1'
 '2_4_2' '2_4_3' '2_4_4' '3_0_0' '3_0_1' '3_0_2' '3_0_3' '3_0_4' '3_1_0'
 '3_1_1' '3_1_2' '3_1_3' '3_1_4' '3_2_0' '3_2_1' '3_2_2' '3_2_3' '3_2_4'
 '3_3_0' '3_3_1' '3_3_2' '3_3_3' '3_3_4' '3_4_0' '3_4_1' '3_4_2' '3_4_3'
 '3_4_4' '4_0_0' '4_0_1' '4_0_2' '4_0_3' '4_0_4' '4_1_0' '4_1_1' '4_1_2'
 '4_1_3' '4_1_4' '4_2_0' '4_2_1' '4_2_2' '4_2_3' '4_2_4' '4_3_0' '4_3_1'
 '4_3_2' '4_3_3' '4_3_4' '4_4_0' '4_4_1' '4_4_2

In [6]:
##### split into N batches of equal M subvolumes each of a given 0_X_X, 1_X_X, ..., 5_X_X
# 6 for TNG50 (216 subvolumes), 5 for TNG100 (125 subvolumes), 7 for TNG300 (343 subvolumes)

Nbatches = 5
subvolume_batches = np.split(subvolume_strs,Nbatches)

print([len(s) for s in subvolume_batches])
print(subvolume_batches)


[25, 25, 25, 25, 25]
[array(['0_0_0', '0_0_1', '0_0_2', '0_0_3', '0_0_4', '0_1_0', '0_1_1',
       '0_1_2', '0_1_3', '0_1_4', '0_2_0', '0_2_1', '0_2_2', '0_2_3',
       '0_2_4', '0_3_0', '0_3_1', '0_3_2', '0_3_3', '0_3_4', '0_4_0',
       '0_4_1', '0_4_2', '0_4_3', '0_4_4'], dtype='<U5'), array(['1_0_0', '1_0_1', '1_0_2', '1_0_3', '1_0_4', '1_1_0', '1_1_1',
       '1_1_2', '1_1_3', '1_1_4', '1_2_0', '1_2_1', '1_2_2', '1_2_3',
       '1_2_4', '1_3_0', '1_3_1', '1_3_2', '1_3_3', '1_3_4', '1_4_0',
       '1_4_1', '1_4_2', '1_4_3', '1_4_4'], dtype='<U5'), array(['2_0_0', '2_0_1', '2_0_2', '2_0_3', '2_0_4', '2_1_0', '2_1_1',
       '2_1_2', '2_1_3', '2_1_4', '2_2_0', '2_2_1', '2_2_2', '2_2_3',
       '2_2_4', '2_3_0', '2_3_1', '2_3_2', '2_3_3', '2_3_4', '2_4_0',
       '2_4_1', '2_4_2', '2_4_3', '2_4_4'], dtype='<U5'), array(['3_0_0', '3_0_1', '3_0_2', '3_0_3', '3_0_4', '3_1_0', '3_1_1',
       '3_1_2', '3_1_3', '3_1_4', '3_2_0', '3_2_1', '3_2_2', '3_2_3',
       '3_2_4', '3_3_0', '3_3_1', 

In [7]:
### finally loop over each batch of subvolumes

# for batchnum in range(len(subvolume_batches)):
for batchnum in range(2):    

    print('===========> working on batch=%s'%batchnum,flush=True)

    jaxify_trees.run(tree_path=tree_path,prefix=prefix,
                     subvolumes=subvolume_batches[batchnum],
                     min_root_logmvir=min_root_logmvir,zs_full=zs_full,min_snaps=min_snaps,
                     out_fname='/mnt/ceph/users/vpandya/sapphire/demo/trees/tng/test_batch%s.npz'%batchnum,
                     params_cosmo=None,num_readers=None)

===========> working on batch=0
since params_cosmo=None, using Planck15
setting num_readers from None to 25


/mnt/home/vpandya/miniconda3/envs/japphiretorch/lib/python3.12/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/mnt/home/vpandya/miniconda3/envs/japphiretorch/lib/python3.12/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Loading tree roots: 100%|██████████| 10214945874/10214945874 [00:16<00:00, 601671542.87it/s]


finished subvolume 0_4_3 in 30.59 sec
finished subvolume 0_4_1 in 43.40 sec
finished subvolume 0_0_4 in 71.74 sec
finished subvolume 0_0_3 in 72.55 sec
finished subvolume 0_1_2 in 74.81 sec
finished subvolume 0_4_2 in 76.97 sec
finished subvolume 0_3_0 in 82.29 sec
finished subvolume 0_4_4 in 85.16 sec
finished subvolume 0_3_1 in 86.71 sec
finished subvolume 0_1_3 in 87.30 sec
finished subvolume 0_0_0 in 103.54 sec
finished subvolume 0_4_0 in 104.24 sec
finished subvolume 0_2_2 in 115.65 sec
finished subvolume 0_2_0 in 143.83 sec
finished subvolume 0_2_4 in 162.41 sec
finished subvolume 0_3_2 in 164.99 sec
finished subvolume 0_1_0 in 174.26 sec
finished subvolume 0_1_4 in 174.98 sec
finished subvolume 0_2_3 in 177.83 sec
finished subvolume 0_0_1 in 180.31 sec
finished subvolume 0_2_1 in 182.22 sec
finished subvolume 0_0_2 in 215.89 sec
finished subvolume 0_3_4 in 220.35 sec
finished subvolume 0_3_3 in 345.01 sec
finished subvolume 0_1_1 in 449.73 sec
finished all in 450.18 sec
saved /m

/mnt/home/vpandya/miniconda3/envs/japphiretorch/lib/python3.12/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Loading tree roots:   0%|          | 0/2140016192 [00:00<?, ?it/s]/mnt/home/vpandya/miniconda3/envs/japphiretorch/lib/python3.12/site-packages/multiprocess/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Loading tree roots: 100%|██████████| 9512991052/9512991052 [00:16<00:00, 590175641.73it/s]
Process ForkPoolWorker-36:
Process ForkPoolWorker-49:


KeyboardInterrupt: 

In [8]:
### test sapphire's jax tree reader on outputs

from sapphire.utils import read_trees

In [9]:
config = {'data_path':'/mnt/ceph/users/vpandya/sapphire/demo/',
          'tree_type':'tng',
          'tree_batches':1,
          'tree_filename':'test_batch%s.npz',
          'Nbatch':None,'rng_halos':3}

In [3]:
rand_halo_matrix, rand_coeff_matrix, rand_halo_tinit, rand_halo_index, ts_interp = read_trees.get(config)

finished reading /mnt/ceph/users/vpandya/sapphire/demo/trees/tng/test_batch0.npz in 0.13 sec
finished reading all batches in 0.13 sec
batch=0 has 34594 halos
total # trees 34594
coeff_matrix.shape (34594, 5, 4, 97)
coeff_matrix.shape after halo mass cuts (34385, 5, 4, 97)
assigning random draw probs took 0.08 sec
requested Nbatch [None] > trees available [34385], using all 34385
rand_coeff_matrix.shape (34385, 5, 4, 97)
rand_halo_index.shape (34385,)
